<a href="https://colab.research.google.com/github/Hem1144/AI-ML/blob/main/MultipleLinearRegressionWeek3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Create Spark session
spark = SparkSession.builder \
    .appName("PropertyPricePrediction") \
    .getOrCreate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = '/content/drive/MyDrive/UEL/property.csv'
property_df = spark.read.csv(file_path, header=True, inferSchema=True)
property_df.show()

+--------------+------------+-------------+----------+--------+------------------+
|Square_Footage|Num_Bedrooms|Num_Bathrooms|Year_Built|Lot_Size|             Price|
+--------------+------------+-------------+----------+--------+------------------+
|          1360|           2|            3|      1953|    7860| 303948.1373854071|
|          4272|           3|            3|      1997|    5292| 860386.2685075302|
|          3592|           4|            1|      1983|    9723| 734389.7538956215|
|           966|           6|            1|      1903|    4086| 226448.8070714377|
|          4926|           6|            4|      1944|    1081|1022486.2616704078|
|          3944|           6|            2|      1938|    3542| 845638.1354384426|
|          3671|           2|            1|      1963|    5105| 748779.2192281872|
|          3419|           4|            2|      1925|    5448| 743007.2614135538|
|           630|           2|            2|      2012|    3204| 135656.4528785377|
|   

In [ ]:
# Define the target variable
target = "Price"

# Define feature combinations to test
feature_combinations = {
    "Model 1": ["Square_Footage", "Num_Bedrooms"],
    "Model 2": ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"],
    "Model 3": ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms", "Year_Built"],
    "Model 4": ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms", "Year_Built", "Lot_Size"],
    "Model 5": ["Square_Footage", "Lot_Size"],
    "Model 6": ["Square_Footage", "Year_Built"]
}

# Dictionary to store R-squared values
r_squared_results = {}

# Train and evaluate each model
for model_name, features in feature_combinations.items():
    print(f"\nTraining {model_name} with features: {features}")

    # Create feature vector
    assembler = VectorAssembler(inputCols=features, outputCol="features")
    assembled_df = assembler.transform(property_df)

    # Split data into training and test sets (70% train, 30% test)
    train_data, test_data = assembled_df.randomSplit([0.7, 0.3], seed=42)

    # Create and train linear regression model
    lr = LinearRegression(featuresCol="features", labelCol=target)
    lr_model = lr.fit(train_data)

    # Make predictions
    predictions = lr_model.transform(test_data)

    # Evaluate model
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="r2")
    r2 = evaluator.evaluate(predictions)

    # Store results
    r_squared_results[model_name] = r2

    print(f"R-squared for {model_name}: {r2:.4f}")
    print(f"Coefficients: {lr_model.coefficients}")
    print(f"Intercept: {lr_model.intercept:.2f}")

# Display all results
print("\nModel Comparison:")
for model, r2 in r_squared_results.items():
    print(f"{model}: R-squared = {r2:.4f}")

# Find the best performing model
best_model = max(r_squared_results, key=r_squared_results.get)
print(f"\nBest performing model is {best_model} with R-squared: {r_squared_results[best_model]:.4f}")


Training Model 1 with features: ['Square_Footage', 'Num_Bedrooms']
R-squared for Model 1: 0.9937
Coefficients: [200.00176646664136,4991.94976774495]
Intercept: 14239.47

Training Model 2 with features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']
R-squared for Model 2: 0.9939
Coefficients: [199.99719882418904,4993.6759853388585,3018.26532221747]
Intercept: 6697.04

Training Model 3 with features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built']
R-squared for Model 3: 0.9941
Coefficients: [199.9956145506279,4992.81398982426,3014.6129569999207,-100.01655140026075]
Intercept: 202800.71

Training Model 4 with features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size']
R-squared for Model 4: 0.9941
Coefficients: [199.99486613877937,4992.722599549324,3014.3893106599103,-100.01982267248503,0.10528001915527856]
Intercept: 202230.80

Training Model 5 with features: ['Square_Footage', 'Lot_Size']
R-squared for Model 5: 0.9927
Coefficients: [20